# 01. 데이터 탐색 (Data Exploration)

**실행 환경:** 로컬 VSCode  
**목적:** 3개 데이터셋 구조 파악, 레이블 분포 분석, 전처리 전략 수립

---

## 데이터셋 개요

| 데이터셋 | 파일 수 | 형식 | 용도 |
|---------|--------|------|------|
| callcenter_qa | 50 | JSON | 분류 모델 학습, RAG 지식베이스 |
| korean_dialogue | 13 | XLSX | 의도 분석, 도메인 분류 |
| llm_instruction | 119,182 | JSON | LLM 파인튜닝 |

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'  # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

In [26]:
# ⭐ 경로 설정 - 네 환경에 맞게 수정됨
BASE_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'raw'
OUTPUT_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'processed'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CALLCENTER_PATH = BASE_PATH / 'callcenter_qa'
DIALOGUE_PATH = BASE_PATH / 'korean_dialogue'
LLM_PATH = BASE_PATH / 'llm_instruction'

print(f"Base path exists: {BASE_PATH.exists()}")
print(f"Callcenter path exists: {CALLCENTER_PATH.exists()}")
print(f"Dialogue path exists: {DIALOGUE_PATH.exists()}")
print(f"LLM path exists: {LLM_PATH.exists()}")

Base path exists: True
Callcenter path exists: True
Dialogue path exists: True
LLM path exists: True


---
## 1. Callcenter QA 데이터 탐색

**스키마:**
- 도메인, 카테고리, 대화셋일련번호, 화자, 문장번호
- 고객의도, 상담사의도, QA
- 고객질문(요청), 상담사질문(요청), 고객답변, 상담사답변
- 개체명, 용어사전, 지식베이스

In [3]:
def load_callcenter_data(path):
    """콜센터 JSON 파일들 로드"""
    all_data = []
    json_files = list(path.rglob('*.json'))
    
    print(f"발견된 JSON 파일 수: {len(json_files)}")
    
    for file in tqdm(json_files, desc='Loading JSON files'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_data.extend(data)
                else:
                    all_data.append(data)
        except Exception as e:
            print(f"Error loading {file.name}: {e}")
    
    return all_data

# 데이터 로드
callcenter_data = load_callcenter_data(CALLCENTER_PATH)
print(f"\n총 레코드 수: {len(callcenter_data):,}")

발견된 JSON 파일 수: 50


Loading JSON files: 100%|██████████| 50/50 [00:16<00:00,  3.03it/s]


총 레코드 수: 2,003,458


In [4]:
# DataFrame 변환
df_callcenter = pd.DataFrame(callcenter_data)

print("=== 컬럼 목록 ===")
print(df_callcenter.columns.tolist())
print(f"\nShape: {df_callcenter.shape}")
df_callcenter.head(3)

=== 컬럼 목록 ===
['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA', '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명 ', '용어사전', '지식베이스']

Shape: (2003458, 15)


,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,K쇼핑,교환,SC8676,상담사,1,,,A,,,,ㅇㅇ쇼핑 ㅇㅇㅇ입니다.,"ㅇㅇ쇼핑, ㅇㅇㅇ",,"ㅇㅇ쇼핑/쇼핑몰명, ㅇㅇㅇ/상담사명"
1,K쇼핑,교환,SC8676,고객,2,교환문의,,Q,네 수고하십니다. 다름이 아니고요. 주문한 게 너무 사이즈가 작아서 바꿀려고 하는데요.,,,,"다름, 주문, 사이즈",사이즈/치수,
2,K쇼핑,교환,SC8676,상담사,3,,고객정보확인,Q,,아 그러세요. 정보 확인 후에 도와 드리겠습니다. 성함하고 전화번호 말씀 부탁 드립...,,,"정보 확인, 성함, 전화번호, 말씀",성함/성명,


In [24]:
# 도메인별 분포 - Plotly 버전
import plotly.express as px
import plotly.graph_objects as go

domain_counts = df_callcenter['도메인'].value_counts()
print("=== 도메인별 분포 ===")
print(domain_counts)

# Plotly 막대 그래프
fig = px.bar(
    x=domain_counts.index,
    y=domain_counts.values,
    labels={'x': '도메인', 'y': '건수'},
    title='도메인별 데이터 분포',
    color=domain_counts.values,
    color_continuous_scale='Blues'
)

fig.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    height=500
)

# 값 표시
fig.update_traces(
    texttemplate='%{y:,.0f}',
    textposition='outside'
)

fig.show()

=== 도메인별 분포 ===
도메인
K쇼핑       1005233
질병관리본부     451865
금융/보험      363641
다산콜센터      182719
Name: count, dtype: int64


In [6]:
# 카테고리별 분포 (상위 20개)
category_counts = df_callcenter['카테고리'].value_counts()
print(f"=== 카테고리 종류: {len(category_counts)}개 ===")
print(category_counts.head(20))

=== 카테고리 종류: 30개 ===
카테고리
결제               239471
업무처리             223199
주문               203086
건강/질병            162511
교환               125820
온라인신고            115444
배송               114223
상품 가입 및 해지        98177
사고 및 보상 문의        94547
증상/징후             91151
이체, 출금, 대출서비스     87258
잔고 및 거래내역         83659
반품                74374
일반행정 문의           56647
생활하수도 관련 문의       45705
대중교통 안내           43829
코로나19 관련 상담       36538
약품/식품             33664
기타문의              23750
요양기관 현황           18178
Name: count, dtype: int64


In [7]:
# QA 분포 확인
qa_counts = df_callcenter['QA'].value_counts()
print("=== Q/A 분포 ===")
print(qa_counts)

# 화자 분포
speaker_counts = df_callcenter['화자'].value_counts()
print("\n=== 화자 분포 ===")
print(speaker_counts)

=== Q/A 분포 ===
QA
Q    1014311
A     989147
Name: count, dtype: int64

=== 화자 분포 ===
화자
고객     1008090
상담사     995368
Name: count, dtype: int64


In [8]:
# 질문(Q) 데이터만 필터링
df_questions = df_callcenter[df_callcenter['QA'] == 'Q'].copy()
print(f"질문(Q) 데이터: {len(df_questions):,}건")

# 답변(A) 데이터
df_answers = df_callcenter[df_callcenter['QA'] == 'A'].copy()
print(f"답변(A) 데이터: {len(df_answers):,}건")

질문(Q) 데이터: 1,014,311건
답변(A) 데이터: 989,147건


In [9]:
# 텍스트 컬럼 확인 - 어떤 컬럼에 실제 텍스트가 있는지
text_columns = ['고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변']

print("=== 텍스트 컬럼별 비어있지 않은 값 수 ===")
for col in text_columns:
    non_empty = df_callcenter[col].fillna('').str.len() > 0
    print(f"{col}: {non_empty.sum():,}건")

=== 텍스트 컬럼별 비어있지 않은 값 수 ===
고객질문(요청): 391,121건
상담사질문(요청): 623,181건
고객답변: 619,455건
상담사답변: 373,021건


In [25]:
# 고객질문 텍스트 길이 분포 - Plotly 버전
import plotly.express as px
import plotly.graph_objects as go

df_callcenter['question_len'] = df_callcenter['고객질문(요청)'].fillna('').str.len()
valid_questions = df_callcenter[df_callcenter['question_len'] > 0]

print("=== 고객질문 텍스트 길이 통계 ===")
print(valid_questions['question_len'].describe())

median_val = valid_questions['question_len'].median()

# Plotly 히스토그램
fig = px.histogram(
    valid_questions,
    x='question_len',
    nbins=50,
    title='고객질문 텍스트 길이 분포',
    labels={'question_len': '문자 수', 'count': '빈도'},
    color_discrete_sequence=['coral']
)

# 중앙값 선 추가
fig.add_vline(
    x=median_val,
    line_dash='dash',
    line_color='red',
    annotation_text=f'중앙값: {median_val:.0f}',
    annotation_position='top'
)

fig.update_layout(
    xaxis_title='문자 수',
    yaxis_title='빈도',
    height=450,
    showlegend=False
)

fig.show()

=== 고객질문 텍스트 길이 통계 ===
count    391121.000000
mean         23.668029
std          14.789395
min           1.000000
25%          15.000000
50%          20.000000
75%          28.000000
max         200.000000
Name: question_len, dtype: float64


In [11]:
# 고객의도 분포
intent_counts = df_callcenter['고객의도'].value_counts()
print(f"=== 고객의도 종류: {len(intent_counts)}개 ===")
print(intent_counts.head(15))

=== 고객의도 종류: 33343개 ===
고객의도
           1474779
코로나 문의        7145
거래내역          7127
상담사의도파악       6822
질병/상해청구       4641
상품가입          4010
이체서비스         3531
방송상품주문        3284
코로나정보         2703
잠시대기요청        2647
버스노선          2597
권한승인요청        2408
자동차보험         2293
분실사고          2281
코로나증상         2232
Name: count, dtype: int64


---
## 2. Korean Dialogue 데이터 탐색

In [12]:
def load_dialogue_data(path):
    """대화 XLSX 파일들 로드"""
    all_dfs = []
    xlsx_files = list(path.rglob('*.xlsx'))
    
    print(f"발견된 XLSX 파일 수: {len(xlsx_files)}")
    
    for file in tqdm(xlsx_files, desc='Loading XLSX files'):
        try:
            df = pd.read_excel(file)
            df['source_file'] = file.stem
            all_dfs.append(df)
        except Exception as e:
            print(f"Error loading {file.name}: {e}")
    
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()

# 데이터 로드
df_dialogue = load_dialogue_data(DIALOGUE_PATH)
print(f"\n총 레코드 수: {len(df_dialogue):,}")

발견된 XLSX 파일 수: 13


Loading XLSX files:  15%|█▌        | 2/13 [00:02<00:12,  1.14s/it]

Error loading l 여권 최종본(250814).xlsx: Excel file format cannot be determined, you must specify an engine manually.


Loading XLSX files:  46%|████▌     | 6/13 [00:05<00:06,  1.06it/s]

Error loading k 상수도_최종본(250814).xlsx: Excel file format cannot be determined, you must specify an engine manually.


Loading XLSX files: 100%|██████████| 13/13 [00:10<00:00,  1.28it/s]

Error loading m 차량등록_최종본(250814).xlsx: Excel file format cannot be determined, you must specify an engine manually.
Error loading j 교통_최종본(250814).xlsx: Excel file format cannot be determined, you must specify an engine manually.

총 레코드 수: 90,413


In [13]:
# 컬럼 확인
print("=== 컬럼 목록 ===")
print(df_dialogue.columns.tolist())
df_dialogue.head(3)

=== 컬럼 목록 ===
['SPEAKER', 'SENTENCE', 'DOMAINID', 'DOMAIN', 'CATEGORY', 'SPEAKERID', 'SENTENCEID', 'MAIN', 'SUB', 'QA', 'QACNCT', 'MQ', 'SQ', 'UA', 'SA', '개체명', '용어사전', '지식베이스', 'source_file', 'Unnamed: 18']


,SPEAKER,SENTENCE,DOMAINID,DOMAIN,CATEGORY,SPEAKERID,SENTENCEID,MAIN,SUB,QA,QACNCT,MQ,SQ,UA,SA,개체명,용어사전,지식베이스,source_file,Unnamed: 18
0,고객,애가 고등학교 1학년인데 태권도 하려면 수강료가 얼마쯤?,C,학원,태권도,1,1,교육비문의,NaN,Q,NaN,애가 고등학교 1학년인데 태권도 하려면 수강료가 얼마쯤?,NaN,NaN,NaN,"고등학교 1학년, 태권도, 수강료, 얼마",NaN,"고등학교 1학년/대상, 태권도/과목","C 학원(4,773)_new",NaN
1,점원,12만 원입니다,C,학원,태권도,0,2,교육비문의,NaN,A,NaN,NaN,NaN,NaN,12만 원입니다,12만 원,NaN,12만 원/금액,"C 학원(4,773)_new",NaN
2,점원,뭐 때문에 하시려는데요?,C,학원,태권도,0,3,상담문의,수강목적,Q,NaN,NaN,뭐 때문에 하시려는데요?,NaN,NaN,NaN,NaN,NaN,"C 학원(4,773)_new",NaN


In [14]:
# 도메인 분포 (korean_dialogue)
if 'DOMAIN' in df_dialogue.columns:
    domain_col = 'DOMAIN'
elif '도메인' in df_dialogue.columns:
    domain_col = '도메인'
else:
    domain_col = None
    print("도메인 컬럼을 찾을 수 없습니다.")

if domain_col:
    dialogue_domain_counts = df_dialogue[domain_col].value_counts()
    print(f"=== 도메인별 분포 (Korean Dialogue) ===")
    print(dialogue_domain_counts)

=== 도메인별 분포 (Korean Dialogue) ===
DOMAIN
의복의류점     15826
음식점       15726
소매        14949
생활서비스     11087
카페         7859
부동산업       7571
숙박         7113
관광여가오락     4949
학원         4773
부동산         560
Name: count, dtype: int64


In [15]:
# MAIN(의도) 분포
if 'MAIN' in df_dialogue.columns:
    main_counts = df_dialogue['MAIN'].value_counts()
    print(f"=== MAIN(의도) 종류: {len(main_counts)}개 ===")
    print(main_counts.head(15))

=== MAIN(의도) 종류: 2540개 ===
MAIN
가격 문의            2982
메뉴문의             1925
제품가격문의           1824
예약문의             1728
가격문의             1558
일반주문             1406
식사주문             1000
상담문의              916
결제요청              868
메뉴주문              577
종류별의류제품문의요청       559
식사배달요청            503
위치문의              482
숙박문의              470
메뉴판에있는음식에대한질문     448
Name: count, dtype: int64


---
## 3. LLM Instruction 데이터 탐색

In [16]:
def load_llm_instruction_sample(path, max_files=100):
    """LLM Instruction JSON 파일 샘플 로드"""
    all_data = []
    json_files = list(path.rglob('*.json'))[:max_files]
    
    print(f"전체 JSON 파일 수: {len(list(path.rglob('*.json'))):,}")
    print(f"샘플 로드: {len(json_files)}개")
    
    for file in tqdm(json_files, desc='Loading LLM samples'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_data.extend(data)
                else:
                    all_data.append(data)
        except:
            pass
    
    return all_data

# 샘플 로드
llm_data = load_llm_instruction_sample(LLM_PATH, max_files=100)
print(f"\n로드된 샘플 레코드 수: {len(llm_data):,}")

전체 JSON 파일 수: 119,182
샘플 로드: 100개


Loading LLM samples: 100%|██████████| 100/100 [00:00<00:00, 1655.84it/s]


로드된 샘플 레코드 수: 100


In [17]:
# LLM 데이터 구조 확인
if llm_data:
    sample = llm_data[0]
    print("=== 샘플 레코드 키 ===")
    for k in sample.keys():
        print(f"  - {k}")
    
    # instructions 구조 확인
    if 'instructions' in sample:
        print(f"\n=== instructions 구조 ===")
        print(f"instructions 개수: {len(sample['instructions'])}")
        if sample['instructions']:
            print(f"첫 번째 instruction 키: {list(sample['instructions'][0].keys())}")

=== 샘플 레코드 키 ===
  - source
  - source_id
  - consulting_category
  - consulting_time
  - consulting_turns
  - consulting_length
  - consulting_content
  - instructions

=== instructions 구조 ===
instructions 개수: 1
첫 번째 instruction 키: ['tuning_type', 'data']


In [18]:
# tuning_type 분포
tuning_types = []
for item in llm_data:
    if 'instructions' in item:
        for inst in item['instructions']:
            if 'tuning_type' in inst:
                tuning_types.append(inst['tuning_type'])

print("=== Tuning Type 분포 ===")
print(Counter(tuning_types))

=== Tuning Type 분포 ===
Counter({'분류': 100})


In [19]:
# consulting_category 분포
categories = [item.get('consulting_category', 'Unknown') for item in llm_data]
cat_counts = Counter(categories)

print(f"=== Consulting Category 종류: {len(cat_counts)}개 ===")
for cat, count in cat_counts.most_common(15):
    print(f"  {cat}: {count}")

=== Consulting Category 종류: 3개 ===
  요금 및 견적: 55
  상품예약 및 결제: 43
  예약변경 및 취소: 2


---
## 4. 데이터 품질 점검

In [20]:
# 결측치 확인 - Callcenter
print("=== Callcenter QA 결측치 ===")
null_counts = df_callcenter.isnull().sum()
null_counts = null_counts[null_counts > 0]
print(null_counts)

=== Callcenter QA 결측치 ===
Series([], dtype: int64)


In [21]:
# 중복 확인
q_col = '고객질문(요청)'
valid_q = df_callcenter[df_callcenter[q_col].fillna('').str.len() > 0]
duplicates = valid_q[q_col].duplicated().sum()
print(f"고객질문 중복: {duplicates:,}건 ({duplicates/len(valid_q)*100:.1f}%)")

고객질문 중복: 81,350건 (20.8%)


---
## 5. 탐색 결과 요약 및 저장

In [22]:
# 탐색 결과 저장
exploration_summary = {
    'callcenter': {
        'total_records': len(df_callcenter),
        'domains': df_callcenter['도메인'].nunique(),
        'categories': df_callcenter['카테고리'].nunique(),
        'intents': df_callcenter['고객의도'].nunique(),
        'questions': len(df_callcenter[df_callcenter['QA'] == 'Q']),
        'answers': len(df_callcenter[df_callcenter['QA'] == 'A']),
        'domain_list': df_callcenter['도메인'].unique().tolist()
    },
    'dialogue': {
        'total_records': len(df_dialogue),
        'columns': df_dialogue.columns.tolist()
    },
    'llm_instruction': {
        'sample_records': len(llm_data),
        'tuning_types': dict(Counter(tuning_types))
    }
}

print("=" * 50)
print("탐색 결과 요약")
print("=" * 50)
print(f"\n[Callcenter QA]")
print(f"  총 레코드: {exploration_summary['callcenter']['total_records']:,}")
print(f"  도메인 수: {exploration_summary['callcenter']['domains']}")
print(f"  카테고리 수: {exploration_summary['callcenter']['categories']}")
print(f"  의도 수: {exploration_summary['callcenter']['intents']}")
print(f"  질문(Q): {exploration_summary['callcenter']['questions']:,}")
print(f"  답변(A): {exploration_summary['callcenter']['answers']:,}")
print(f"\n[Korean Dialogue]")
print(f"  총 레코드: {exploration_summary['dialogue']['total_records']:,}")
print(f"\n[LLM Instruction]")
print(f"  샘플 레코드: {exploration_summary['llm_instruction']['sample_records']:,}")
print(f"  Tuning Types: {exploration_summary['llm_instruction']['tuning_types']}")

탐색 결과 요약

[Callcenter QA]
  총 레코드: 2,003,458
  도메인 수: 4
  카테고리 수: 30
  의도 수: 33343
  질문(Q): 1,014,311
  답변(A): 989,147

[Korean Dialogue]
  총 레코드: 90,413

[LLM Instruction]
  샘플 레코드: 100
  Tuning Types: {'분류': 100}


In [23]:
# JSON으로 저장
with open(OUTPUT_PATH / 'exploration_summary.json', 'w', encoding='utf-8') as f:
    json.dump(exploration_summary, f, ensure_ascii=False, indent=2)

print(f"\n탐색 결과 저장: {OUTPUT_PATH / 'exploration_summary.json'}")


탐색 결과 저장: /Users/kuka/CIVILCOMPLAINT/data/processed/exploration_summary.json


---
## 다음 단계

→ **02_preprocessing.ipynb** 에서:
1. 텍스트 정제 (특수문자, 개인정보 마스킹)
2. 레이블 통합 매핑
3. Train/Val/Test 분할
4. 전처리된 데이터 저장